In [1]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

1 Introduction

GeoJSON to RDF Knowledge Graph

This notebook demonstrates how OpenStreetMap (OSM) vector data (ways) can be transformed into an RDF knowledge graph using Python.

It covers the full workflow from loading GeoJSON data with GeoPandas to generating RDF triples with rdflib, including GeoSPARQL geometries and Turtle serialization.


* 2 Load GeoJSON 
* 3 Prepare CRS
* 4 Build RDF Graph
* 5 Add GeoSPARQL Geometries
* 6 Export Turtle
* 7 Example RDF Output


In [2]:
# Import packages
import geopandas as gpd
from pathlib import Path
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, SKOS

In [3]:
# Namespaces
AGRO = Namespace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#")
GEO  = Namespace("http://www.opengis.net/ont/geosparql#")

In [21]:
# Import Data

BASE_DIR = Path.cwd().parent  # eine Ebene nach oben

OUTPUT_DIR = BASE_DIR / "data" / "processed"

INPUT_FILE = BASE_DIR / "data" / "osm" / "ways_osm.geojson"

OUTPUT_FILE = BASE_DIR / "data" / "processed" / "agro_individuals_with_geom.ttl"



In [12]:
# Set CRS
gdf = gpd.read_file(INPUT_FILE)

if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(4326)

In [13]:
# Initial Graph

g = Graph()

g.bind("agro", AGRO)
g.bind("geo", GEO)
g.bind("skos", SKOS)
g.bind("rdfs", RDFS)

In [14]:
# Creating RDF

for idx, row in gdf.iterrows():

    way_uri  = AGRO[f"osm_way_{idx+1}"]
    geom_uri = AGRO[f"geom_osm_way_{idx+1}"]

    # Klasse
    g.add((way_uri, RDF.type, AGRO.OSMWay))

    # Label
    g.add((way_uri, SKOS.prefLabel,
           Literal(f"OSM Way {idx+1}", lang="en")))

    # Geometry
    g.add((geom_uri, RDF.type, GEO.Geometry))
    g.add((geom_uri, GEO.asWKT,
           Literal(row.geometry.wkt,
           datatype=GEO.wktLiteral)))

    g.add((way_uri, GEO.hasGeometry, geom_uri))

In [22]:
g.serialize(destination=OUTPUT_FILE, format="turtle")

print("Features:", len(gdf))
print("Triples:", len(g))
print("Saved:", OUTPUT_FILE)

Features: 2537
Triples: 12685
Saved: c:\Users\frank\Desktop\agro-geokg\data\processed\agro_individuals_with_geom.ttl
